<a href="https://colab.research.google.com/github/rai8896/Deep_Learning_Lab/blob/main/Experiment_9_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wandb huggingface_hub -q

import wandb
wandb.login()

from huggingface_hub import notebook_login
notebook_login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

from torch.utils.data import DataLoader, random_split
from huggingface_hub import HfApi, create_repo

HF_REPO = "mraidtu/exp9-gan-dcgan"
WANDB_PROJECT = "exp9-gan-dcgan"

EPOCHS = 4
BATCH_SIZE = 128
LATENT_DIM = 100
IMAGE_SIZE = 28
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOSSES = ["bce", "lsgan", "wgan"]
OPTIMIZERS = ["adam", "sgd", "rmsprop"]
MODELS = ["gan", "dcgan"]

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1]
])

dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class Generator_GAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator_GAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Generator_DCGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(LATENT_DIM, 128, 7, 1, 0),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z.view(-1, LATENT_DIM, 1, 1))


class Discriminator_DCGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def get_loss(name):
    if name == "bce":
        return nn.BCELoss()
    elif name == "lsgan":
        return nn.MSELoss()
    elif name == "wgan":
        return None

In [ ]:
def get_optimizer(params, name):
    if name == "adam":
        return optim.Adam(params, lr=0.0002, betas=(0.5, 0.999))
    elif name == "sgd":
        return optim.SGD(params, lr=0.01)
    else:
        return optim.RMSprop(params, lr=0.0002)

In [ ]:
def run_experiment(model_type, loss_type, opt_name):
    name = f"{model_type}_{loss_type}_{opt_name}"
    print(f"\nStarting: {name}")

    wandb.init(project=WANDB_PROJECT, name=name)

    if model_type == "gan":
        G = Generator_GAN().to(DEVICE)
        D = Discriminator_GAN().to(DEVICE)
    else:
        G = Generator_DCGAN().to(DEVICE)
        D = Discriminator_DCGAN().to(DEVICE)

    criterion = get_loss(loss_type)

    opt_G = get_optimizer(G.parameters(), opt_name)
    opt_D = get_optimizer(D.parameters(), opt_name)

    for epoch in range(EPOCHS):
        for real, _ in train_loader:
            real = real.to(DEVICE)
            batch = real.size(0)

            # =================================================
            # 🔥 Train Discriminator
            # =================================================
            opt_D.zero_grad()

            z = torch.randn(batch, LATENT_DIM).to(DEVICE)
            fake = G(z)

            if loss_type == "wgan":
                loss_D = -(torch.mean(D(real)) - torch.mean(D(fake)))
            else:
                real_labels = torch.ones(batch, 1).to(DEVICE)
                fake_labels = torch.zeros(batch, 1).to(DEVICE)

                loss_real = criterion(D(real), real_labels)
                loss_fake = criterion(D(fake.detach()), fake_labels)
                loss_D = loss_real + loss_fake

            loss_D.backward()
            opt_D.step()

            # =================================================
            # 🔥 Train Generator (FIXED)
            # =================================================
            opt_G.zero_grad()

            # IMPORTANT FIX 🔥 (new fake sample)
            z = torch.randn(batch, LATENT_DIM).to(DEVICE)
            fake = G(z)

            if loss_type == "wgan":
                loss_G = -torch.mean(D(fake))
            else:
                real_labels = torch.ones(batch, 1).to(DEVICE)
                loss_G = criterion(D(fake), real_labels)

            loss_G.backward()
            opt_G.step()

        wandb.log({
            "epoch": epoch,
            "G_loss": loss_G.item(),
            "D_loss": loss_D.item()
        })

        print(f"Epoch {epoch}: G={loss_G:.4f} D={loss_D:.4f}")

    torch.save(G.state_dict(), f"{name}_G.pt")
    wandb.finish()

    return G

In [ ]:
results = {}
saved_models = {}

for model in MODELS:
    for loss in LOSSES:
        for opt in OPTIMIZERS:
            G = run_experiment(model, loss, opt)
            saved_models[f"{model}_{loss}_{opt}"] = G


Starting: gan_bce_adam


Epoch 0: G=8.7552 D=0.9892
Epoch 1: G=4.9983 D=0.9470
Epoch 2: G=4.2857 D=0.4257
Epoch 3: G=2.0739 D=0.2925


D_loss,██▂▁
G_loss,█▄▃▁
epoch,▁▃▆█
D_loss,0.29252
G_loss,2.07393
epoch,3



Starting: gan_bce_sgd


Epoch 0: G=2.0498 D=0.2795
Epoch 1: G=4.0833 D=0.1260
Epoch 2: G=4.4692 D=0.0418
Epoch 3: G=2.8006 D=0.1315


...

Epoch 0: G=-0.0020 D=-0.9942
Epoch 1: G=-0.0010 D=-0.9977
Epoch 2: G=-0.0005 D=-0.9986
Epoch 3: G=-0.0006 D=-0.9994


D_loss,█▃▂▁
G_loss,▁▆██
epoch,▁▃▆█
D_loss,-0.99942
G_loss,-0.00057
epoch,3



Starting: dcgan_wgan_rmsprop


Epoch 0: G=-0.3480 D=-0.5451
Epoch 1: G=-0.0504 D=-0.6600
Epoch 2: G=-0.2033 D=-0.4549
Epoch 3: G=-0.4128 D=-0.5426


D_loss,▅▁█▅
G_loss,▂█▅▁
epoch,▁▃▆█
D_loss,-0.54255
G_loss,-0.41281
epoch,3


In [ ]:
create_repo(HF_REPO, exist_ok=True)
api = HfApi()

for name in saved_models:
    fname = f"{name}_G.pt"
    if os.path.exists(fname):
        api.upload_file(
            path_or_fileobj=fname,
            path_in_repo=f"models/{fname}",
            repo_id=HF_REPO
        )
        print(f"uploaded: {fname}")



Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  gan_bce_adam_G.pt           :  25%|##4       |  556kB / 2.24MB            

uploaded: gan_bce_adam_G.pt


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

uploaded: dcgan_wgan_sgd_G.pt


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  dcgan_wgan_rmsprop_G.pt     :  92%|#########1| 2.79MB / 3.05MB            

uploaded: dcgan_wgan_rmsprop_G.pt


In [ ]:
username = wandb.api.viewer()['entity']
print(f"W&B : https://wandb.ai/{username}/{WANDB_PROJECT}")
print(f"HF : https://huggingface.co/{HF_REPO}")

W&B : https://wandb.ai/manishrai_25afi23-delhi-technological-university/exp9-gan-dcgan
HF : https://huggingface.co/mraidtu/exp9-gan-dcgan
